# Examen del primer bimestre: recuperacion de informacion

Implementacion y evaluacion de un motor lexico para SciFact. Se comparan TF-IDF con coseno y BM25 sobre documentos completos; no se usan embeddings.

**Preprocesamiento compartido con `app.py`:** texto en minusculas, tokenizacion con `[a-z]+`, stopwords inglesas de NLTK, descarte de tokens de longitud <= 2 y stemming Porter.


## 1. Preparacion del entorno y rutas

In [19]:

# %pip install -r requirements.txt

from pathlib import Path
import math, re, html, os
from collections import Counter, defaultdict
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data/scifact/corpus/corpus.parquet").exists() and (ROOT / "Examen/data/scifact/corpus/corpus.parquet").exists():
    ROOT = ROOT / "Examen"
DATA = ROOT / "data/scifact"
nltk.download("stopwords", quiet=True)
STOP = set(stopwords.words("english"))
STEM = PorterStemmer()
print("Directorio de datos:", DATA.resolve())


Directorio de datos: C:\Users\Kevin\Downloads\Proyecto (1)\Examen\data\scifact


## 2. Carga de corpus, consultas y juicios de relevancia

In [20]:
corpus_df = pd.read_parquet(DATA / "corpus/corpus.parquet")
queries_df = pd.read_parquet(DATA / "queries/queries.parquet")

def read_qrels(path):
    df = pd.read_csv(path, sep="\t")
    # Los archivos oficiales pueden traer encabezados query-id/corpus-id o query_id/doc_id.
    aliases = {"query-id":"query_id", "corpus-id":"doc_id"}
    df = df.rename(columns=aliases)
    if not {"query_id", "doc_id", "score"}.issubset(df.columns):
        df = pd.read_csv(path, sep="\t", header=None, names=["query_id", "doc_id", "score"])
    qrels = defaultdict(set)
    for row in df.itertuples(index=False):
        if int(row.score) > 0:
            qrels[str(row.query_id)].add(str(row.doc_id))
    return dict(qrels)

docs, titles = {}, {}
for row in corpus_df.itertuples(index=False):
    doc_id = str(getattr(row, "_0", None) or corpus_df.loc[row.Index, "_id"]) if hasattr(row, "Index") else None
# Iteracion explicita para preservar los identificadores originales.
for _, row in corpus_df.iterrows():
    doc_id = str(row["_id"])
    titles[doc_id] = html.unescape(str(row["title"] or ""))
    docs[doc_id] = (titles[doc_id] + " " + html.unescape(str(row["text"] or ""))).strip()
queries = {str(row["_id"]): str(row["text"]) for _, row in queries_df.iterrows()}
train_path, test_path = ROOT / "train.tsv", ROOT / "test.tsv"
# Preferimos las copias entregadas en la raiz de Examen; fallback a qrels SciFact del dataset.
train_qrels = read_qrels(train_path) if train_path.exists() else {}
test_qrels = read_qrels(test_path) if test_path.exists() else read_qrels(DATA / "qrels/test.tsv")
print(f"{len(docs)} documentos, {len(queries)} consultas, {len(train_qrels)} qrels train, {len(test_qrels)} qrels test")


5183 documentos, 1109 consultas, 809 qrels train, 300 qrels test


## 3. Preprocesamiento e indice invertido

In [21]:
def preprocesar(texto):
    tokens = re.findall(r"[a-z]+", str(texto).lower())
    return [STEM.stem(t) for t in tokens if t not in STOP and len(t) > 2]

def construir_indice(documentos):
    postings, longitudes = defaultdict(dict), {}
    for doc_id, texto in documentos.items():
        tokens = preprocesar(texto)
        longitudes[doc_id] = len(tokens)
        for token, freq in Counter(tokens).items():
            postings[token][doc_id] = freq
    n = len(documentos)
    idx = {"indice": dict(postings), "len": longitudes, "N": n, "avgdl": sum(longitudes.values()) / n}
    normas = defaultdict(float)
    for token, lista in idx["indice"].items():
        idf = math.log(1 + n / len(lista))
        for d, f in lista.items(): normas[d] += ((1 + math.log(f)) * idf) ** 2
    idx["normas"] = {d: math.sqrt(v) for d, v in normas.items()}
    return idx

idx = construir_indice(docs)
print(f"Vocabulario: {len(idx['indice']):,} terminos; longitud media: {idx['avgdl']:.1f} tokens")


Vocabulario: 20,614 terminos; longitud media: 135.3 tokens


## 4. Modelos de recuperacion: TF-IDF y BM25

In [22]:
def tfidf(query, idx):
    q = Counter(preprocesar(query)); scores = defaultdict(float); norm_q = 0.0
    for term, fq in q.items():
        posting = idx["indice"].get(term)
        if not posting: continue
        idf = math.log(1 + idx["N"] / len(posting)); wq = (1 + math.log(fq)) * idf
        norm_q += wq ** 2
        for d, f in posting.items(): scores[d] += wq * (1 + math.log(f)) * idf
    norm_q = math.sqrt(norm_q) or 1.0
    return {d: s / (norm_q * idx["normas"][d]) for d, s in scores.items() if idx["normas"].get(d, 0)}

def bm25(query, idx, k1=1.5, b=0.75):
    scores = defaultdict(float)
    for term in set(preprocesar(query)):
        posting = idx["indice"].get(term)
        if not posting: continue
        df = len(posting); idf = math.log(1 + (idx["N"] - df + 0.5) / (df + 0.5))
        for d, f in posting.items():
            denom = f + k1 * (1 - b + b * idx["len"][d] / idx["avgdl"])
            scores[d] += idf * f * (k1 + 1) / denom
    return dict(scores)

def buscar(model, query, k=10):
    scores = bm25(query, idx) if model == "bm25" else tfidf(query, idx)
    return sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]

consulta_demo = next(iter(queries.values()))
for model in ("tfidf", "bm25"):
    print(model, buscar(model, consulta_demo, 3))


tfidf [('13231899', 0.13728072235258404), ('21257564', 0.07877703800413807), ('31543713', 0.07844282644720817)]
bm25 [('26071782', 10.066764070044272), ('21257564', 9.945339319785328), ('40212412', 9.346379170376828)]


## 5. Metricas de evaluacion

In [23]:
def metricas(ranking, relevantes, k=10):
    ranking_k = ranking[:k]
    hits = [1 if d in relevantes else 0 for d in ranking_k]
    precision = sum(hits) / k
    recall = sum(hits) / len(relevantes) if relevantes else 0.0
    ap_sum = 0.0; encontrados = 0
    for rank, hit in enumerate(hits, 1):
        if hit:
            encontrados += 1; ap_sum += encontrados / rank
    ap = ap_sum / min(len(relevantes), k) if relevantes else 0.0
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, 1))
    ideal_hits = min(len(relevantes), k)
    idcg = sum(1 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return {f"Precision@{k}": precision, f"Recall@{k}": recall, f"MAP@{k}": ap, f"nDCG@{k}": dcg / idcg if idcg else 0.0}

def evaluar(qrels, split, k=10):
    rows = []
    for model in ("tfidf", "bm25"):
        per_query = []
        for qid, relevant in qrels.items():
            if qid not in queries or not relevant: continue
            ranking = [d for d, _ in buscar(model, queries[qid], k)]
            per_query.append(metricas(ranking, relevant, k))
        if per_query:
            rows.append({"split": split, "model": model, **{m: sum(x[m] for x in per_query) / len(per_query) for m in per_query[0]}, "n_queries": len(per_query)})
    return pd.DataFrame(rows)

train_results = evaluar(train_qrels, "train", k=10) if train_qrels else pd.DataFrame()
test_results = evaluar(test_qrels, "test", k=10)
results = pd.concat([train_results, test_results], ignore_index=True)
display(results.round(4))


,split,model,Precision@10,Recall@10,MAP@10,nDCG@10,n_queries
0,train,tfidf,0.0889,0.7961,0.6150,0.6633,809
1,train,bm25,0.0902,0.8070,0.6448,0.6885,809
2,test,tfidf,0.0897,0.8125,0.5734,0.6357,300
3,test,bm25,0.0893,0.8120,0.6341,0.6811,300


## 6. Comparacion y conclusiones

In [24]:
test_table = results[results["split"] == "test"].set_index("model")
if {"tfidf", "bm25"}.issubset(test_table.index):
    display(test_table[["Precision@10", "Recall@10", "MAP@10", "nDCG@10"]].round(4))
    best = test_table["nDCG@10"].idxmax()
    print(f"En test, {best.upper()} obtiene el mayor nDCG@10. Usa estos resultados medidos para redactar la conclusion.")
else:
    print("No hay consultas compartidas con qrels para comparar; revisa los identificadores y archivos TSV.")

print("Interpretacion: TF-IDF usa pesos logaritmicos y similitud coseno; BM25 incorpora saturacion de frecuencia y normalizacion por longitud. La ventaja concreta se determina con las metricas calculadas, no se presupone.")


,Precision@10,Recall@10,MAP@10,nDCG@10
model,,,,
tfidf,0.0897,0.8125,0.5734,0.6357
bm25,0.0893,0.8120,0.6341,0.6811


En test, BM25 obtiene el mayor nDCG@10. Usa estos resultados medidos para redactar la conclusion.
Interpretacion: TF-IDF usa pesos logaritmicos y similitud coseno; BM25 incorpora saturacion de frecuencia y normalizacion por longitud. La ventaja concreta se determina con las metricas calculadas, no se presupone.


## 7. Prueba del servicio desplegado

In [25]:
import requests

# Sustituye esta URL por la URL publica real de Render, sin barra final.
SERVICE_URL = os.getenv("SEARCH_SERVICE_URL", "https://recuperacion-de-la-informacion.onrender.com")
if "TU-SERVICIO" in SERVICE_URL:
    print("Configura SEARCH_SERVICE_URL con la URL publicada para ejecutar esta demostracion.")
else:
    health = requests.get(f"{SERVICE_URL}/health", timeout=30)
    print("GET /health:", health.status_code, health.json())
    payload = {"query": consulta_demo, "model": "bm25", "top_k": 5}
    response = requests.post(f"{SERVICE_URL}/search", json=payload, timeout=60)
    print("POST /search:", response.status_code)
    display(pd.DataFrame(response.json().get("results", [])))


GET /health: 200 {'documents_indexed': 5183, 'message': 'Servicio de busqueda activo', 'status': 'ok'}
POST /search: 200


,document_id,rank,score,snippet,title
0,26071782,1,10.066764,"Latent membrane protein 1 (LMP1), an oncoprote...",Latent membrane protein 1 of Epstein–Barr viru...
1,21257564,2,9.945339,The paramount feature of long-term potentiatio...,How long will long-term potentiation last?
2,40212412,3,9.346379,Life forms that have low body mass can hunt fo...,Periosteal bone formation--a neglected determi...
3,18953920,4,8.800135,The epithelial-mesenchymal transition (EMT) is...,The Epithelial-Mesenchymal Transition Generate...
4,34386619,5,8.613907,The Bacillus subtilis clpC operon is regulated...,The first gene of the Bacillus subtilis clpC o...


## 8. Resumen

Se construyo un indice invertido sobre documentos completos de SciFact y se evaluaron los rankings de TF-IDF coseno y BM25 con Precision@10, Recall@10, MAP@10 y nDCG@10. Se reportan por separado train y test cuando ambos archivos de qrels estan disponibles. La celda anterior permite validar el mismo endpoint de busqueda que se prueba con Postman.